# 🚀 BÁO CÁO: NHẬN DIỆN DEEPFAKE BẰNG MÔ HÌNH LSDA (CVPR 2024)
**Paper:** Transcending Forgery Specificity with Latent Space Augmentation for Generalizable Deepfake Detection (Yan et al., CVPR 2024 / arXiv:2311.11278)

Notebook này bao gồm toàn bộ quy trình: Tải Data, Cắt mặt MTCNN, Build mô hình Teacher-Student LSDA, Huấn luyện chống OOM và Đánh giá chuẩn Bài báo Khoa học.

## 1. Cài đặt Môi trường

In [ ]:
!pip install timm facenet-pytorch Pillow==9.5.0 -q
print("✅ Cài đặt môi trường thành công!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, random, time, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm
from sklearn.metrics import roc_curve, auc, accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE} | Torch: {torch.__version__}")

## 2. Kết nối Dataset FaceForensics++ (C23)

In [ ]:
import kagglehub

print("Đang tải dataset FaceForensics++ C23...")
dataset_path = kagglehub.dataset_download("xdxd003/ff-c23")

# Tự động dò thư mục gốc
BASE = dataset_path
if 'original' not in os.listdir(BASE):
    if 'FaceForensics++_C23' in os.listdir(BASE):
        BASE = os.path.join(BASE, 'FaceForensics++_C23')
    else:
        for root, dirs, _ in os.walk(dataset_path):
            if 'original' in dirs:
                BASE = root
                break

FAKE_DIRS = {
    'Deepfakes':      os.path.join(BASE, 'Deepfakes'),
    'Face2Face':      os.path.join(BASE, 'Face2Face'),
    'FaceSwap':       os.path.join(BASE, 'FaceSwap'),
    'NeuralTextures': os.path.join(BASE, 'NeuralTextures'),
}
REAL_DIR = os.path.join(BASE, 'original')
print("✅ Đã trỏ chính xác vào tập dữ liệu FF++!")

## 3. Tiền Xử Lý: Trích xuất khuôn mặt (MTCNN) + Smart Backup
Quá trình này tốn khoảng 15 phút. Nếu bạn đã từng chạy mô hình UCF và lưu ảnh vào thư mục `UCF_frames_backup` trên Drive, hệ thống sẽ tự động TÁI SỬ DỤNG toàn bộ mà không cần cắt lại!

In [ ]:
OUT_DIR = '/content/frames'
# Dùng chung thư mục backup với UCF để tiết kiệm dung lượng và thời gian!
BACKUP_DIR = '/content/drive/MyDrive/UCF_frames_backup' 
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

MAX_VIDEOS_PER_CLASS = 1000   
FRAMES_PER_VIDEO     = 10     

folders = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
mtcnn = MTCNN(margin=20, keep_all=False, post_process=False, device=DEVICE)

def extract_faces_mtcnn(video_list, save_path):
    os.makedirs(save_path, exist_ok=True)
    count = 0
    for vid_path in tqdm(video_list, desc=f"Cắt -> {os.path.basename(save_path)}"):
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); continue
        target_frames = set([int(i * total_frames / FRAMES_PER_VIDEO) for i in range(FRAMES_PER_VIDEO)])
        vid_name = os.path.basename(vid_path).split('.')[0]
        cur = 0; saved = 0
        while True:
            ret = cap.grab()
            if not ret: break
            if cur in target_frames:
                ret, frame = cap.retrieve()
                if ret:
                    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    face = mtcnn(img_pil)
                    if face is not None:
                        face_bgr = cv2.cvtColor(face.permute(1,2,0).cpu().numpy().astype(np.uint8), cv2.COLOR_RGB2BGR)
                        cv2.imwrite(os.path.join(save_path, f"{vid_name}_{saved}.jpg"), face_bgr)
                        count += 1
                saved += 1
                if saved >= FRAMES_PER_VIDEO: break
            cur += 1
        cap.release()
    return count

for folder in folders:
    src_folder = os.path.join(OUT_DIR, folder)
    bup_folder = os.path.join(BACKUP_DIR, folder)
    
    if not os.path.exists(src_folder) and os.path.exists(bup_folder):
        print(f"♻️ Khôi phục '{folder}' từ Drive...")
        shutil.copytree(bup_folder, src_folder)
    elif os.path.exists(src_folder) and len(os.listdir(src_folder)) > 100:
        print(f"✅ Thư mục '{folder}' đã sẵn sàng.")
    else:
        print(f"=== ĐANG CẮT ẢNH CHO '{folder}' ===")
        if folder == 'original':
            vids = [os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR) if f.endswith('.mp4')]
        else:
            vids = [os.path.join(FAKE_DIRS[folder], f) for f in os.listdir(FAKE_DIRS[folder]) if f.endswith('.mp4')]
        random.shuffle(vids)
        extract_faces_mtcnn(vids[:MAX_VIDEOS_PER_CLASS], src_folder)
        print(f"📦 Backup '{folder}' lên Drive...")
        shutil.copytree(src_folder, bup_folder, dirs_exist_ok=True)

print("🎉 HOÀN TẤT BƯỚC TIỀN XỬ LÝ ẢNH!")

## 4. Chuẩn bị DataLoader (Hyperparameters chuẩn LSDA)

In [ ]:
MANIP2IDX = {'Deepfakes': 0, 'Face2Face': 1, 'FaceSwap': 2, 'NeuralTextures': 3}

def collect_images(folder, label_binary, label_manip):
    paths = []
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.lower().endswith('.jpg'):
                paths.append((os.path.join(folder, f), label_binary, label_manip))
    return paths

all_samples = []
all_samples += collect_images(os.path.join(OUT_DIR, 'original'), 0, 4)
for name in FAKE_DIRS.keys():
    all_samples += collect_images(os.path.join(OUT_DIR, name), 1, MANIP2IDX[name])

random.shuffle(all_samples)
n = len(all_samples)
assert n > 0, "❌ Không tìm thấy ảnh!"

n_train = int(0.8 * n); n_val = int(0.1 * n)
train_s = all_samples[:n_train]; val_s = all_samples[n_train:n_train+n_val]; test_s = all_samples[n_train+n_val:]
print(f"✅ Tổng ảnh: {n} | Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

# --- THÔNG SỐ HUẤN LUYỆN LSDA ---
IMG_SIZE   = 224    # Kích thước chuẩn của LSDA (ResNet)
BATCH_SIZE = 32     # Phù hợp GPU T4/L4
EPOCHS     = 30     # Chuẩn CVPR 2024
LR         = 2e-4
FEAT_DIM   = 512

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class FFDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lb, lm = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), lb, lm

train_loader = DataLoader(FFDataset(train_s, train_tf), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FFDataset(val_s,   val_tf),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FFDataset(test_s,  val_tf),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 5. Kiến trúc LSDA Model (Teacher-Student & Latent Space Augmentation)
Giữ nguyên bản chuẩn paper CVPR 2024: Teacher ResNet50, Student ResNet34.

In [ ]:
class TeacherEncoder(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        base = timm.create_model('resnet50', pretrained=True, num_classes=0)
        self.backbone = base
        self.proj = nn.Linear(2048, feat_dim)
    def forward(self, x): return self.proj(self.backbone(x))

class StudentEncoder(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        base = timm.create_model('resnet34', pretrained=True, num_classes=0)
        self.backbone = base
        self.proj = nn.Linear(512, feat_dim)
    def forward(self, x): return self.proj(self.backbone(x))

class LSDAModel(nn.Module):
    def __init__(self, feat_dim=512, num_domains=4):
        super().__init__()
        self.teacher = TeacherEncoder(feat_dim)
        self.student = StudentEncoder(feat_dim)
        self.domain_heads = nn.ModuleList([nn.Linear(feat_dim, feat_dim) for _ in range(num_domains+1)])
        self.domain_cls = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_domains+1))
        self.binary_cls = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))

    def latent_augment(self, feats, labels, alpha=0.4):
        B = feats.size(0)
        lam = torch.FloatTensor(B).uniform_(alpha, 1-alpha).to(feats.device)
        idx = torch.randperm(B).to(feats.device)
        lam_feat = lam.view(-1, 1)  
        aug_feats = lam_feat * feats + (1 - lam_feat) * feats[idx]
        return aug_feats, labels, labels[idx], lam  

    def forward(self, x, labels=None, augment=False):
        t_feat = self.teacher(x)
        domain_logit = self.domain_cls(t_feat)
        s_feat = self.student(x)
        aug_f = lb_a = lb_b = lam = None
        if augment and labels is not None:
            aug_f, lb_a, lb_b, lam = self.latent_augment(s_feat, labels)
            bin_logit = self.binary_cls(aug_f).squeeze(1)
        else:
            bin_logit = self.binary_cls(s_feat).squeeze(1)
        return bin_logit, domain_logit, t_feat, s_feat, aug_f, lb_a, lb_b, lam

model = LSDAModel(feat_dim=FEAT_DIM, num_domains=4).to(DEVICE)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Khởi tạo xong Model LSDA | Params: {n_p/1e6:.1f}M")

## 6. Training Pipeline & Loss Functions (MixUp + Distillation)

In [ ]:
bce_fn = nn.BCEWithLogitsLoss(reduction='none') 
bce_scalar = nn.BCEWithLogitsLoss()             
ce_fn = nn.CrossEntropyLoss()

def mixup_loss(logits, y_a, y_b, lam):
    loss_a = bce_fn(logits, y_a.float())  
    loss_b = bce_fn(logits, y_b.float())  
    return (lam * loss_a + (1 - lam) * loss_b).mean()  

def distill_loss(t_feat, s_feat, T=4.):
    t_norm = F.normalize(t_feat.detach() / T, dim=1)
    s_norm = F.normalize(s_feat / T, dim=1)
    return F.mse_loss(s_norm, t_norm)

L_BIN = 1.0; L_DOMAIN = 0.5; L_DISTIL = 0.3

teacher_p = list(model.teacher.parameters())
student_p = (list(model.student.parameters()) + list(model.domain_heads.parameters()) 
             + list(model.domain_cls.parameters()) + list(model.binary_cls.parameters()))

optimizer = optim.AdamW([{'params': teacher_p, 'lr': LR*0.1}, {'params': student_p, 'lr': LR}], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def train_epoch(model, loader, optimizer):
    model.train()
    tot_loss = 0; correct = 0; total = 0
    for imgs, lb, lm in loader:
        imgs = imgs.to(DEVICE); lb = lb.to(DEVICE); lm = lm.to(DEVICE)
        optimizer.zero_grad()
        domain_lb = torch.where(lb == 0, torch.full_like(lm, 4), lm)
        bin_logit, dom_logit, t_feat, s_feat, aug_f, lb_a, lb_b, lam_mix = model(imgs, labels=lb, augment=True)
        
        l_bin = mixup_loss(bin_logit, lb_a, lb_b, lam_mix) if aug_f is not None else bce_scalar(bin_logit, lb.float())
        l_dom = ce_fn(dom_logit, domain_lb)
        l_dis = distill_loss(t_feat, s_feat)
        
        loss = L_BIN*l_bin + L_DOMAIN*l_dom + L_DISTIL*l_dis
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        tot_loss += loss.item() * imgs.size(0)
        with torch.no_grad():
            cl = model.binary_cls(s_feat).squeeze(1)
            correct += (torch.sigmoid(cl) > 0.5).long().eq(lb).sum().item()
        total += imgs.size(0)
    torch.cuda.empty_cache() # Chống văng RAM
    return tot_loss / total, correct / total

def eval_epoch(model, loader):
    model.eval()
    tot_loss = 0; all_p = []; all_l = []
    with torch.no_grad():
        for imgs, lb, lm in loader:
            imgs = imgs.to(DEVICE); lb = lb.to(DEVICE); lm = lm.to(DEVICE)
            domain_lb = torch.where(lb == 0, torch.full_like(lm, 4), lm)
            bin_logit, dom_logit, t_feat, s_feat, *_ = model(imgs, augment=False)
            
            l_bin = bce_scalar(bin_logit, lb.float())
            l_dom = ce_fn(dom_logit, domain_lb)
            l_dis = distill_loss(t_feat, s_feat)
            
            tot_loss += (L_BIN*l_bin + L_DOMAIN*l_dom + L_DISTIL*l_dis).item() * imgs.size(0)
            all_p.extend(torch.sigmoid(bin_logit).cpu().numpy())
            all_l.extend(lb.cpu().numpy())
    probs = np.array(all_p); labels = np.array(all_l)
    acc = accuracy_score(labels, (probs > 0.5).astype(int))
    fpr, tpr, _ = roc_curve(labels, probs)
    return tot_loss / len(labels), acc, auc(fpr, tpr), probs, labels, fpr, tpr

## 7. Thực thi quá trình huấn luyện

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive'
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_auc = 0
history  = {'tr_loss': [], 'vl_loss': [], 'tr_acc': [], 'vl_acc': [], 'vl_auc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc              = train_epoch(model, train_loader, optimizer)
    vl_loss, vl_acc, vl_auc, *_ = eval_epoch(model, val_loader)
    scheduler.step()
    
    for k, v in zip(history.keys(), [tr_loss, vl_loss, tr_acc, vl_acc, vl_auc]): history[k].append(v)
    
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'lsda_best.pth'))
        tag = " ← 🏆 BEST"
    else: tag = ""
    print(f"Epoch {epoch:02d} | Loss: {tr_loss:.4f}/{vl_loss:.4f} | Acc: {tr_acc:.4f}/{vl_acc:.4f} | AUC: {vl_auc:.4f} | {time.time()-t0:.0f}s{tag}")

## 8. Đánh giá chất lượng (Academic Quality Plots)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'font.family': 'sans-serif', 'figure.dpi': 300, 'savefig.dpi': 300, 'axes.linewidth': 1.5, 'lines.linewidth': 2.5})
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.set_palette("husl")

axes[0].plot(history['tr_loss'], label='Train Loss', alpha=0.9); axes[0].plot(history['vl_loss'], label='Val Loss', alpha=0.9)
axes[0].set_title('Loss Evolution', fontweight='bold'); axes[0].legend(frameon=True, shadow=True)

axes[1].plot(history['tr_acc'], label='Train Acc', alpha=0.9); axes[1].plot(history['vl_acc'], label='Val Acc', alpha=0.9)
axes[1].set_title('Accuracy Evolution', fontweight='bold'); axes[1].legend(frameon=True, shadow=True)

axes[2].plot(history['vl_auc'], label='Val AUC', color='#2ca02c', alpha=0.9)
axes[2].set_title('AUC Evolution', fontweight='bold'); axes[2].legend(frameon=True, shadow=True)

plt.tight_layout(pad=2.0)
plt.savefig(os.path.join(OUTPUT_DIR, 'lsda_training_curves.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 9. Đánh giá Tối hậu trên tập Test (Kèm EER Score)

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'lsda_best.pth'), map_location=DEVICE))
_, test_acc, test_auc, probs, labels, fpr, tpr = eval_epoch(model, test_loader)
eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

print(f"==================================================")
print(f"📊 IN-DATASET EVALUATION (FaceForensics++ C23)")
print(f"🎯 Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"🎯 AUC      : {test_auc:.4f}")
print(f"🎯 EER      : {eer:.4f}")
print(f"==================================================")

plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'LSDA (AUC = {test_auc:.4f} | EER = {eer:.4f})')
plt.plot([0,1],[0,1], color='navy', lw=2, linestyle='--', alpha=0.6)
plt.scatter([eer], [1-eer], s=200, marker='*', color='gold', edgecolor='black', zorder=5, label='Equal Error Rate')
plt.xlim([-0.02, 1.02]); plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontweight='bold'); plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC)\nFaceForensics++ C23', fontweight='bold', pad=15)
plt.legend(loc='lower right', frameon=True, shadow=True, fontsize=12)
plt.savefig(os.path.join(OUTPUT_DIR, 'lsda_roc_ffpp.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 10. Cross-Dataset Evaluation (Celeb-DF-v2)

In [ ]:
# Hướng dẫn test chéo trên Celeb-DF (Đúng tiêu chuẩn bài báo LSDA)
"""
# BƯỚC 1: Tải bộ dữ liệu Celeb-DF-v2 và cắt ảnh tương tự như FF++
# BƯỚC 2: Khởi tạo DataLoader

CELEB_DF_DIR = '/content/drive/MyDrive/CelebDF_frames'
celeb_samples = []
celeb_samples += collect_images(os.path.join(CELEB_DF_DIR, 'real'), label_binary=0, label_manip=4)
celeb_samples += collect_images(os.path.join(CELEB_DF_DIR, 'fake'), label_binary=1, label_manip=0) 
celeb_loader = DataLoader(FFDataset(celeb_samples, val_tf), BATCH_SIZE, shuffle=False, num_workers=2)

# BƯỚC 3: Đánh giá bằng LSDA Model đã train
_, celeb_acc, celeb_auc, probs, labels, fpr, tpr = eval_epoch(model, celeb_loader)
celeb_eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

print(f"📊 CROSS-DATASET EVALUATION (Celeb-DF v2)")
print(f"🎯 Accuracy : {celeb_acc:.4f} | AUC: {celeb_auc:.4f} | EER: {celeb_eer:.4f}")
"""
print("✅ Cấu trúc Test chéo (Cross-Dataset) đã sẵn sàng. Hãy bỏ comment khi bạn có ảnh Celeb-DF!")